# Importing Libraries


In [4]:
import time
import json
import re
from typing import Any, Dict, List, Optional, Tuple
import requests
import pandas as pd


# Config

In [5]:
BASE_URL = "https://api.inaturalist.org/v1/observations"

PROJECT_ID = 41347
PER_PAGE = 200

# You can keep your fields param if you want, but many folks find it inconsistent across iNat endpoints.
# Safer: request full observations and just parse what we need.
USE_FIELDS_PARAM = False

FIELDS_PARAM_VALUE = (
    "id,uri,ofvs,taxon,photos,geojson,location,positional_accuracy,"
    "place_country_name,place_state_name,place_county_name,place_town_name"
)

# Rate limiting (be nice to the API)
SLEEP_SECONDS = 1.0

# Output
OUT_CSV = "inat_project_41347_observations.csv"

# Helper Functions

In [6]:
def safe_get(d: Any, path: List[Any], default=None):
    """Safely get nested keys/lists: path like ['taxon','name'] or ['photos',0,'url']"""
    cur = d
    for p in path:
        try:
            if isinstance(p, int):
                cur = cur[p]
            else:
                cur = cur.get(p)
        except Exception:
            return default
        if cur is None:
            return default
    return cur

In [7]:
def norm(s: str) -> str:
    """Normalize string for fuzzy matching."""
    if s is None:
        return ""
    return re.sub(r"\s+", " ", str(s).strip().lower())


def json_list_str(x: List[Any]) -> str:
    """Store lists in CSV safely as JSON strings."""
    return json.dumps(x, ensure_ascii=False)

# Fetch observations with pagination

In [9]:
def fetch_all_observations(
    project_id: int,
    per_page: int = 200,
    use_fields_param: bool = False,
    fields_value: Optional[str] = None,
    sleep_seconds: float = 1.0,
) -> List[Dict[str, Any]]:
    session = requests.Session()
    session.headers.update({
        # A descriptive UA is polite; add your email/affiliation if you want.
        "User-Agent": "inat-project-export"
    })

    page = 1
    all_results: List[Dict[str, Any]] = []

    while True:
        params = {
            "project_id": project_id,
            "per_page": per_page,
            "page": page,
        }
        if use_fields_param and fields_value:
            params["fields"] = fields_value

        resp = session.get(BASE_URL, params=params, timeout=60)

        if resp.status_code == 429:
            wait = max(5, int(sleep_seconds * 5))
            print(f"Rate limited (429). Sleeping {wait}s then retrying page {page}...")
            time.sleep(wait)
            continue

        resp.raise_for_status()
        data = resp.json()

        results = data.get("results", [])
        total = data.get("total_results", None)

        if not results:
            break

        all_results.extend(results)
        print(f"Fetched page {page} | got {len(results)} | total so far {len(all_results)}" + (f" / {total}" if total else ""))

        # Stop condition based on total_results if present
        if total is not None and page * per_page >= total:
            break

        page += 1
        time.sleep(sleep_seconds)

    return all_results

# Extract observation

In [11]:
def extract_ofv_value(ofvs: List[Dict[str, Any]], target_name: str) -> Optional[str]:
    """
    Find an OFV by 'name' (case-insensitive). If multiple, join unique values with '; '.
    """
    if not ofvs:
        return None

    t = norm(target_name)
    values = []
    for item in ofvs:
        name = norm(item.get("name", ""))
        if name == t:
            val = item.get("value")
            if val is not None and str(val).strip():
                values.append(str(val).strip())
    # de-dup preserving order
    seen = set()
    out = []
    for v in values:
        if v not in seen:
            out.append(v)
            seen.add(v)
    return "; ".join(out) if out else None


In [12]:
def extract_ofv_value_fuzzy(ofvs: List[Dict[str, Any]], contains_text: str) -> Optional[str]:
    """
    Fuzzy match: returns joined values where ofv.name contains contains_text.
    Useful if the field label varies slightly.
    """
    if not ofvs:
        return None
    t = norm(contains_text)
    values = []
    for item in ofvs:
        name = norm(item.get("name", ""))
        if t in name:
            val = item.get("value")
            if val is not None and str(val).strip():
                values.append(str(val).strip())

    # de-dup
    seen = set()
    out = []
    for v in values:
        if v not in seen:
            out.append(v)
            seen.add(v)
    return "; ".join(out) if out else None

# Photos: collect URL lists

In [13]:
def extract_photo_urls(photos: List[Dict[str, Any]]) -> Dict[str, List[str]]:
    """
    Returns lists of urls per size when present.
    We store: square, medium, original (when available).
    """
    out = {"square": [], "medium": [], "original": []}
    if not photos:
        return out

    for p in photos:
        # Common iNat keys seen in responses:
        # url (often square), square_url, medium_url, original_url, large_url
        square = p.get("square_url") or p.get("url")
        medium = p.get("medium_url")
        original = p.get("original_url")

        if square:
            out["square"].append(square)
        if medium:
            out["medium"].append(medium)
        if original:
            out["original"].append(original)

        # If original_url isn't present, you can often derive it by removing `/square` or `/medium`
        # BUT not always reliable, so we only store it when explicitly provided.

    return out

# Taxonomy extraction